# 08. 강화학습 기초

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/%EC%8B%A4%EC%8A%B5_08_%EA%B0%95%ED%99%94%ED%95%99%EC%8A%B5_%EA%B8%B0%EC%B4%88.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## Actor--Critic: 정책과 가치의 역할 분리

### 🔬 실습 — 한 스텝 Actor--Critic 손실 계산


In [ ]:
import torch
import torch.nn.functional as F

# 한 배치의 예시 출력
logits = torch.tensor([[0.2, 1.0, -0.3]], requires_grad=True)
value = torch.tensor([0.6], requires_grad=True)
next_value = torch.tensor([0.8])
action = torch.tensor([1])
reward = torch.tensor([0.5])
done = torch.tensor([0.0])
gamma = 0.99

logp = F.log_softmax(logits, dim=-1).gather(1, action[:, None]).squeeze(1)
td_target = reward + gamma * next_value * (1 - done)
advantage = td_target - value

actor_loss = -(logp * advantage.detach()).mean()
critic_loss = 0.5 * advantage.pow(2).mean()
entropy = -(F.softmax(logits, -1) * F.log_softmax(logits, -1)).sum(-1).mean()
loss = actor_loss + critic_loss - 0.01 * entropy
loss.backward()

print("TD target:", td_target.item())
print("advantage:", advantage.item())
print("actor/critic:", actor_loss.item(), critic_loss.item())

## 실습: 그리드월드 테이블 Q-learning

### 🔬 실습 — 그리드월드에서 Q-learning으로 경로 학습

① **Q-테이블 초기화**: 상태 수 × 행동 수(4) 크기의 표를 0으로 시작.


In [ ]:
import numpy as np, random
def run_qlearning(env, episodes=2000, alpha=0.1, gamma=0.99,
                  eps_start=1.0, eps_end=0.05, eps_decay=0.995):
    Q = np.zeros((env.n * env.n, 4))    # (상태, 행동)
    eps, returns = eps_start, []
    for ep in range(episodes):
        sid, total = env.state_id(env.reset()), 0.0
        for _ in range(200):
            if random.random() < eps:
                a = random.randrange(4)            # 탐색
            else:
                a = int(np.argmax(Q[sid]))         # 활용
            s2, r, done = env.step(a)
            s2id = env.state_id(s2)
            td_target = r + gamma * np.max(Q[s2id]) * (1 - done)
            Q[sid, a] += alpha * (td_target - Q[sid, a])
            sid, total = s2id, total + r
            if done: break
        eps = max(eps_end, eps * eps_decay)        # epsilon decay
        returns.append(total)
    return Q, returns

② **ε-greedy 행동 선택**: 확률 eps로 무작위, 아니면 Q 최댓값 행동.


③ **Q-learning 업데이트**: TD 타깃에 max_a Q(s',a)를 사용(off-policy).


④ **학습 실행**: 장애물이 있는 6×6 격자에서 2000 에피소드 학습.


**GridWorld 환경 정의** — 위 `run_qlearning`이 사용할 격자 환경입니다.

In [ ]:
# GridWorld: (0,0)에서 출발해 (n-1,n-1) 목표에 도달하는 격자 환경.
# 장애물은 통과 불가(제자리). state_id는 (행,열)을 정수 상태로 변환.
class GridWorld:
    def __init__(self, n=6, obstacles=None):
        self.n = n
        self.obstacles = set(obstacles or [])
        self.start = (0, 0)
        self.goal = (n - 1, n - 1)
        self.pos = self.start
    def reset(self):
        self.pos = self.start
        return self.pos
    def state_id(self, s):
        return s[0] * self.n + s[1]          # (행,열) -> 0..n*n-1
    def step(self, a):
        r0, c0 = self.pos
        dr, dc = [(-1, 0), (1, 0), (0, -1), (0, 1)][a]   # 상/하/좌/우
        r, c = r0 + dr, c0 + dc
        if not (0 <= r < self.n and 0 <= c < self.n) or (r, c) in self.obstacles:
            r, c = r0, c0                    # 벽/장애물이면 제자리
        self.pos = (r, c)
        if self.pos == self.goal:
            return self.pos, 1.0, True       # 목표 도달: +1, 종료
        return self.pos, -0.01, False        # 한 걸음 비용

In [ ]:
env = GridWorld(n=6, obstacles=[(1,1),(1,2),(2,4),(3,1),(4,3)])
Q, returns = run_qlearning(env, episodes=2000)
print("초기 100ep 평균 보상:", np.mean(returns[:100]).round(2))
print("마지막 100ep 평균 보상:", np.mean(returns[-100:]).round(2))

---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
